# Introduction to Decision Trees: Trying to Predict Which NFL Players Will Be An All-Pro

Welcome! Feel free to edit, play around, and modify these cells to your heart's content—they are for your own experimentation and discovery!

# I. Introduction

What is a better way to kick off the 2026-2027 NFL season with a bit of NFL-related data modeling! For those that are unfamiliar, at the culmination of each NFL season, a series of sports writers, pundits, and other people who spend far too much analyzing, writing, and scrutinizing grown men play a ball sport determine which players at each position were the best, a designation known as **All-Pro**. Now this might seem just like a nice title to have, but this has real financial implications; players often have lucrative incentives in their contract for becoming an All-Pro, they are deciding factors in whether a player will make the Pro Football Hall of Fame, and of course, a focal part of a player's legacy. Thus, it serves to see which types of players become All-Pros and see if we can figure it out before those pretentious sports folks!

They are a lot of nuances in how All-Pros are determined, however for our purposes, we are going to focus on the All-Pros as determined by the **Associated Press**. The Associated Press awards two designations: **First-team All-Pro** and **Second-team All-Pro**, with the former being the absolute best at each position and the latter acting as the runner-up. Also, as you will see later in the notebook, we are going to narrow our focus to a single position to make our modeling process a bit more robust and comprehensive. 

A key focus of this demo will demonstrate that most of work behind machine learning is **not dictated by the complexity of your models** (though that can certainly have a major impact). You will actually spend most of your time modeling **playing around with your data** and ensuring that is filtered down, formatted, and standardized so that your models have the best chance of succeeding. This is why having skills in tools such as Pandas and NumPy are so important; models can change, data manipulation skills will not!

![Image of the Best Quarterback in the NFL, Drake Maye](img/i_love_drake_maye.jpg)

# II. Getting Our Data

Throughout this demo, we will be using data from two primary sources:
- [`nflreadpy`](https://github.com/nflverse/nflreadpy): The Python import of the incredibly popular `nflreadr` package. For a long time, R was the defacto programming language for a lot of visualization and data work with Python, but with packages like this, Python is slowly catching up. With this, we can pull play-by-play data, season statistics, advanced NextGenStats, and a ton of more information!
- CSV exports from [`pro-football-reference.com`](https://www.pro-football-reference.com): Pro-football-reference.com is one of the most popular archives of NFL data around, and has been that way for decades. However, they understandably don't have an API and have severe restrictions around web scraping to protect the information they work so hard to obtain. However, they do have static exports of their data that we can painstankingly download and save. Luckily for you, I already did most of this work for you to get which players were All-Pros since 2015. All of this data can be found in the `data` folder. 

In [ ]:
# Most of you likely won't have these packages installed, so let's download this:
!pip install nflreadpy pyarrow polars plotly

# If you need some of the other dependencies we are working with
# !pip install pandas scikit-learn numpy matplotlib 

In [ ]:
# Our imports!
import pandas as pd
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import plotly.express as px
import matplotlib.pyplot as plt
import nflreadpy as nfl
import glob

## II.I Getting our Player Statistic Data

Let's load in the stats for the seasons we are looking at (since 2010), both the play-by-play stats and also some advanced analytics.

I highly encourage you to reference the nflreadpy package as you do this, with documentation [available here](https://nflreadpy.nflverse.com)

In [ ]:
# Let's first get the necessary data we need from nflreadpy

# This is to get player stats from 2010. 

# We could do it like this:
# player_stats = nfl.load_player_stats([2010, 2011, 2012, 2013,...])

# But that is so slow! Let's do this instead:


In [ ]:
# Let's take a look at our data!


In [ ]:
# We can start manipulating this how we want and exploring for example:


In [ ]:
# Let's get a sense of our data at a macro-level
display()

print(f"The number of rows in our Player Stats Data: {player_stats.shape[0]:,}")
print(f"The number of columns in our Census Data: {player_stats.shape[1]}")

# Print our columns in a nice 5x5 list
print(
    f"These are the columns within our dataset: \n{'\n'.join([f'{', '.join(player_stats.columns[i:i+5])}' for i in range(0, len(player_stats.columns), 5)])}"
)

In [ ]:
# Instead, we can narrow this down to just receivers. 
# We also need to set our index to `player_display_name` and `season` as these are unique values 


# Let's check the updated counts of our dataframe
print(f"The number of rows in our Player Stats Data: {receiver_stats.shape[0]:,}")
print(f"The number of columns in our Census Data: {receiver_stats.shape[1]}")
receiver_stats.head(5)

In [ ]:
# Let's select the important columns we need for receivers
# Let's select the important columns we need for receivers
# These are the columns:
"""
    "player_id",
    "season",
    "player_display_name",
    "targets",
    "receptions",
    "receiving_yards",
    "receiving_tds",
    "receiving_air_yards",
    "receiving_yards_after_catch",
    "receiving_fumbles_lost",
    "receiving_epa",
    "racr", # This is a ratio for the number of receiving yards per air yards targeted per game
    "target_share",
    "wopr"  # Weighted Opportunity Rating (combines target share and air yard share into one metric)
"""

receiver_stats.head(5)

In [ ]:
# We can also find data from next-gen stats to add an extra level of nuance to our data

next_gen_stats.head(5)

In [ ]:
# Our data is given to us per-week, but we can find regular season summary by filtering week == 0
next_gen_stats = 

# Some of these columns overlap with our other dataset, so let's just keep the things we need
next_gen_stats = next_gen_stats[[
    "player_gsis_id",
    "player_display_name",
    "season",
    "avg_cushion",
    "avg_separation",
    "avg_intended_air_yards",
    "percent_share_of_intended_air_yards",
    "catch_percentage",
    "avg_yac",
    "avg_expected_yac",
    "avg_yac_above_expectation",
]]

next_gen_stats.head(5)

We want to join these two dataframes together, which we can do through the very powerful `pd.DataFrame.merge()` method! Merges & Joins are arguably the most powerful tool in your data manipulation arsenal and can allow you to combine multiple datasets together in highly-complex and nuanced ways. 

*What is the difference between a `join` and a `merge`? You can find out more [here](https://www.shanelynn.ie/merge-join-dataframes-python-pandas-index-1/)*

We'll do a `left` merge to combine `receiving_stats` with `next_gen_stats`, using the `player_gsis_id` column as our key. This means every row from `receiving_stats` will be kept, and we'll add matching info from `next_gen_stats` where it exists.

![Gif of a Left Join](https://miro.medium.com/v2/resize:fit:1344/1*a3qrRVc7b5VddMTsHLm2dw.gif) \
*Courtesy of [this Medium Article](https://medium.com/analytics-vidhya/everything-about-pandas-100-code-snippets-and-tricks-443e5ae59e81) by Senthil E*

In [ ]:
# Let's perform our join
# Question: Why is it problematic if we join just on the name of each player?
combined_receiver_stats: pd.DataFrame = receiver_stats.merge(
    right = , # The other dataframe we would like to merge on
    left_on = [], # The names of the column in the left dataframe we would like to merge on
    right_on = [], # The names of the column in the right dataframe we would like to merge on
    how = "left", # The type of join we would like to do
)

display(combined_receiver_stats)

print(
    f"These are the columns within our dataset: \n{'\n'.join([f'{', '.join(combined_receiver_stats.columns[i:i+5])}' for i in range(0, len(combined_receiver_stats.columns), 5)])}"
)

In [ ]:
# Some column cleanup after our merge
combined_receiver_stats = combined_receiver_stats.drop(columns=['player_gsis_id', 'player_display_name_y'])
combined_receiver_stats = combined_receiver_stats.rename(columns={'player_display_name_x': 'player_display_name'})

# Not all of our receivers have advanced NextGenStats data, so we ought to remove them
combined_receiver_stats = 

combined_receiver_stats

# II.II: Loading Our All-Pro Data

Take a look at the `data` folder that is within this repository, you should see a bunch of files that look like:

- `all-pro-players-2010.csv`
- `all-pro-players-2011.csv`
- `all-pro-players-2012.csv`
- ...

Now, we need to load all these *distinct* CSV files into Pandas; how do we do that?

In [ ]:
"""
We could do this:
all_pro_players_2010 = pd.read_csv("data/all-pro-players-2010.csv")
all_pro_players_2011 = pd.read_csv("data/all-pro-players-2011.csv)

But that will take forever! Let's find another way
"""

To accomplish this, we will take advantage of a nifty computational operator called a **wildcard operator** or **\***. Suppose you have a bunch of files that look like:

- `file_1`
- `file_2`
- `file_3`
- `not_a_file_1`
- `not_a_file_2`

And we only want the things that begin with `file`. A wildcard operator written like `file-*` will be able to find those files effortlessly without having to laboriously specify each distinct file by name. Pretty neat!

To do this in Python, we will use that `glob` dependency that we imported earlier. Let's take a look:

In [ ]:
# Using glob and the wildcard operator, we can find all of the CSV files that match our `all-pro-players` format
path_to_our_data = "data/all-pro-players-*.csv"

try:
    all_pro_files = glob.glob(path_to_our_data)
    if not all_pro_files:
        raise FileNotFoundError(
            f"No files found matching the pattern {path_to_our_data}. "
            "Please check that the data directory and all-pro CSV files exist."
        )
except Exception as e:
    print(f"Error occurred while searching for files: {e}")
    all_pro_files = []

print(all_pro_files)

In [ ]:
# Let's convert all of this data into Pandas Dataframes
all_pro_players = [pd.read_csv(all_pro_data) for all_pro_data in all_pro_files]

In [ ]:
# Right now, these are all distinct dataframes, let's combine them together for easy filtering

display(all_pro_players.head(5))
print(
    f"These are the columns within our dataset: \n{'\n'.join([f'{', '.join(all_pro_players.columns[i:i+5])}' for i in range(0, len(all_pro_players.columns), 5)])}"
)

In [ ]:
# Let's do some similar steps to what we do above for the player stats: just get the wide receivers and just select the columns we need
all_pro_receivers: pd.DataFrame = all_pro_players.loc[
    (all_pro_players["Pos"] == "WR") & (all_pro_players["Year"] >= seasons[0])
].reset_index()

all_pro_receivers = all_pro_receivers[["Player", "All-pro teams", "Year"]]

all_pro_receivers.head(5)

This is great! But remember, earlier we said we are just focused on All-Pros as designated by the Associated Press (AP). However, this dataset includes people from different sources (such as Pro Football Focus, PFF, and Pro Football Writers of America, FW). To get around this, we will need to do some more filtering to just extract the AP designations and what specifically those designations are (1st Team or 2nd Team). 

In [ ]:
# We need to do per-column string filtering in Pandas, which can be a little weird at first but is very powerful!
all_pro_receivers['is_ap_in_column'] = all_pro_receivers['All-pro teams'].str.contains('AP:')

all_pro_receivers

In [ ]:
# Great! Now once we know if AP is in a column, we can take this a step further and extract the text if AP exists
# To do this, we will use Regular Expressions (RegEx), which is a very common string pattern matching tool 
all_pro_receivers['ap_designation'] = all_pro_receivers['All-pro teams'].str.extract(r'AP: (.{6})')

all_pro_receivers.head(10)

In [ ]:
# Let's filter down our dataset now to just the receivers that were designated an All-Pro by the Associated Press
print(f"We currently have {len(all_pro_receivers)} receivers in our dataset")
all_pro_receivers_ap = all_pro_receivers.loc[
    all_pro_receivers["ap_designation"].notna()
][["Player", "Year", "ap_designation"]]
display(all_pro_receivers_ap.head(10))
print(f"We currently have {len(all_pro_receivers_ap)} receivers in our dataset")

## II.III Combining our Data Together

Now that we have both our datasets, we can combine them together and get to some modeling afterwards!

In [ ]:
# Just for standardization, I am going to create temporary versions of our player name columns to ensure that there is no funky capitalization
combined_receiver_stats['name_standard'] = combined_receiver_stats['player_display_name'].str.lower()
all_pro_receivers_ap['name_standard'] = all_pro_receivers_ap['Player'].str.lower()

In [ ]:
# Let's combine our columns together!
df: pd.DataFrame = combined_receiver_stats.merge(
    right = , # The other dataframe we would like to merge on 
    left_on = [], # The names of the column in the left dataframe we would like to merge on
    right_on = [], # The names of the column in the right dataframe we would like to merge on
    how = "left" # The type of merge
)

# Let's drop some of these extra columns we don't need
df = df.drop(columns=["Player", "Year", "name_standard"])

# Let's fill in Players Who are Not All Pros with a more descriptive name
df['ap_designation'] = df['ap_designation'].fillna('Not AP')

In [ ]:
df

With all of your data combined, we have successfully created a rich dataset that is well-suited for modeling. We have a series of independent variables (our receiving stats) and dependent variable (AP Designation) to now begin modeling. But, before we go there, let's visualize our data to make sure we fully understand the implications of modeling.

![Summary of our Data Pre-Processing Steps](./img/our_data_fig.png)

# II. Visualizations

In [ ]:
# Really quick, let's get a sense of our data before modeling by creating some quick visualizations!

# First, the receivers with the most receiving yards in our dataset
# Important: the same player can appear in multiple seasons, so if we plot just on
# player name, Plotly treats those rows as one x-category and stacks the bars.
# Make each bar unique by combining name + season.
top_receivers: pd.DataFrame = df.sort_values(by="receiving_yards", ascending=False).head(15).copy()
top_receivers["player_season"] = (
    top_receivers["player_display_name"]
    + " ("
    + top_receivers["season"].astype(str)
    + ")"
)
display(top_receivers.head(3))

fig: px.bar = px.bar(
    top_receivers,
    x="player_season",
    y="receiving_yards",
    title="Top 15 Receivers by Receiving Yards",
    labels={
        "player_season": "Player (Season)",
        "receiving_yards": "Receiving Yards",
    },
)
fig.update_layout(xaxis_tickangle=45)
fig.show()


In [ ]:
# Very important for modeling: the distribution of our classes
# Class imbalance is one of the largest problems with classification
# Ideally classes are balanced, but All-Pros are rare by design!

class_counts = df["ap_designation"].value_counts()

display(class_counts)


fig.update_xaxes(tickangle=30)
fig.show()


In [ ]:
# Let's take a peek at something different: how the average receiving EPA has changed over the years
receiving_epa_grouped_by_year = df.groupby("season")["receiving_epa"].agg(
    ["count", "mean"]
).reset_index()

display(receiving_epa_grouped_by_year)

fig = px.line(
    receiving_epa_grouped_by_year, 
    x="season", 
    y="mean", 
    markers=True,
    title="Average Receiving EPA by Season",
    labels={"season": "Season", "mean": "Mean Receiving EPA"},
    line_shape='linear'
)
fig.update_traces(line_color="steelblue")
fig.update_xaxes(dtick=1)
fig.update_layout(height=400, width=800)
fig.show()

# III. Decision Tree Modeling

# III.I Data Splitting

Before modeling, we need to divide our dataset into training and testing datasets. As a reminder:
- **Training**: What our model will see and use to determine the decision boundaries
- **Testing**: What we will use to evaluate the accuracy and performance of our model
To do this, let's take all of ours rows before 2024 as our training data and all of our data after 2024 as our testing

Typically while modeling, we will use Sci-Kit Learn's built-in function called `train_test_split` to randomly split the data into training and testing sets. Random splitting is generally the norm for most machine learning problems, helping to avoid leaks of information from the future into the training set and ensuring robust performance metrics.

However, in this specific case, because our dataset is time-series based (players across seasons), we want to more realistically simulate prediction into the future. So instead of using random splitting, we take all rows before 2024 as training data (what our model "knows" about the past), and all data from 2024 and onward as test data (simulating predictions on new, unseen seasons). This chronological split better matches how our model would be used in realistic scenarios: learning from the past, then making predictions on subsequent seasons.

If you wanted to do a random split, you could use:

```python
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["ap_designation"]  # keep class distribution balanced if needed
)
```

In [ ]:
# Sort our data chronologically
sorted_df: pd.DataFrame = df.sort_values("season").reset_index(drop=True)

train_df = 
test_df = 

In [ ]:
X_trn = train_df.drop(
    columns=["player_id", "player_display_name", "season", "ap_designation"]
)
y_trn = train_df["ap_designation"]
X_tst = test_df.drop(
    columns=["player_id", "player_display_name", "season", "ap_designation"]
)
y_tst = test_df["ap_designation"]

## III.II: Default Decision Tree

In [ ]:
# Let's make a very basic decision tree, with real no tuning of parameters
model = 

In [ ]:
# Let's fit our model to our training data


In [ ]:
# Let's use our fitted model our testing data!


In [ ]:
# Let's see our model is performing


In [ ]:
# What are the actual decisions that our decision made? We can investigate that!
plot_tree(
    model,
    filled=True,
    feature_names=X_trn.columns,
    class_names=tuned_model.classes_.astype(str),
    rounded=True,
    fontsize=7
)
plt.title("Basic Decision Tree")
plt.tight_layout()
plt.show()

### What are all these metrics??

When we use a decision tree (or any classifier), it's important to understand *how* the model is performing.
Let's make sense of the main metrics you'll see in the classification report:

- **Accuracy**: The simplest metric — it tells us what proportion of predictions our model got right overall.
  > *"Out of all samples, how many did we classify correctly?"*
  - **This seems like the easiest? Why don't we just always use it?**
    - Accuracy can be really misleading!
    - For example, if only a few players each year make an All-Pro team, a model that just predicts "Not All-Pro" for everyone will have high accuracy, but in reality will be a terrible model because it didn't identify any of our 1st or 2nd Team players!
    - Thus, we need some other metrics to see how our models perform on all classes of data, even those that might be rarer 

- **Precision**: For a given class, precision is the fraction of cases we predicted as that class that were actually correct.
  > *"When the model predicts a player is '1st Team', how often is it right?"*
  - We are looking to minimize *false positives*: the model saying a player is a particular class, when in reality they are not

- **Recall** (also called Sensitivity): Recall is the fraction of all actual positive cases (e.g., real '1st Team' All-Pros) that our model correctly identified.
  > *"Of all real '1st Team' All-Pros, how many did our model find?"*
  - We are looking to minimize *false negatives*: the model not correctly identifying a player is of a particular class

- **F1 Score**: The F1 Score balances precision and recall (it's the harmonic mean of the two). It gives a single score that rewards models that achieve both high precision *and* high recall.
  > *"How good is the model at both finding a class and being correct when it does?"*


## III.III: Tuned Decision Tree

Let's try to introduce a bit more detail and nuance into our classification process to see if we can get some better results

In [ ]:
# We will specify quite a few more parameters here to see if that improves performance
tuned_model: DecisionTreeClassifier = DecisionTreeClassifier(
    max_depth=5,            # Stops the tree from growing too deep and complex
    min_samples_split=15,   # Requires at least 20 rows to justify making a new split
    min_samples_leaf=10,    # Ensures every final "guess" is based on at least 10 rows
    class_weight='balanced', # Penalize our model for misclassifying minority classes
    random_state=42
)

In [ ]:
# Fit our tuned model


In [ ]:
# Use this (hopefully) improved model on the testing data


In [ ]:
# Basic decision tree plotting using scikit-learn's built-in plot_tree
plot_tree(
    tuned_model,
    filled=True,
    feature_names=X_trn.columns,
    class_names=tuned_model.classes_.astype(str),
    rounded=True,
    fontsize=7
)
plt.title("Tuned Decision Tree")
plt.tight_layout()
plt.show()

In [ ]:
# Let's see what our tuned model predicted the 2024-2025 1st-Team All-Pros to be!
prediction_comparison = test_df
prediction_comparison['model_answer'] = y_pred_tuned
prediction_comparison.loc[prediction_comparison['model_answer'] != prediction_comparison['ap_designation']]

# Where do we go from here?
This is truly just scraping the iceberg of predictive modeling and there are countless opportunities to extend this analysis further depending on what your interests are?

- **Interested in more data manipulation work?**: Try to change the position from wide receivers to another position (QB, CB, etc...), go through the process of identifying key variables based upon documentation, and also ensure that all of our data is properly formatted based on your desired model
  - Some models required [standardization or normalization](https://www.datacamp.com/tutorial/normalization-vs-standardization), so make sure you are putting the right format of data into your models!
- **Interested more in the modeling portion?**: Try [Random Forests](https://inria.github.io/scikit-learn-mooc/python_scripts/ensemble_random_forest.html), which are an *ensemble* of multiple decision trees together that can provide more robust reasoning capabilities or [XGBoost](https://medium.com/@imoisharma18/gentle-introduction-of-xgboost-library-2b1ac2669680), which build multiple decision trees sequentially and gradually work to improve the model via Gradient Descent
- Both of these models, while more powerful, will require the optimization of **hyperparameters**, which are the specific knobs that influence how a machine learning model performs. To learn more, look into [hyperparameter optimization](https://inria.github.io/scikit-learn-mooc/python_scripts/ensemble_hyperparameters.html)
- **Hate this dataset and never want to see it again?**: Totally fair! However, you can take the same ideas of data manipulation and modeling to really any dataset or topic, and the principles should transfer over quite seamlessly. If you need some help, feel free to reach out!